# Taxonomy Comparison Scorecard

Use this notebook to compare **previous** vs **new** categorization outputs across:

1. Full dataset (distribution / stability)
2. `PROPOSED_L3`-style backtest subset (agreement + category migrations)
3. SME sample (ground-truth quality metrics)

Outputs are saved as CSV/JSON artifacts for easy sharing.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 180)

In [ ]:
# -----------------------------
# CONFIG: paths + column names
# -----------------------------
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Input files (set these to your actual files)
FULL_DATA_PATH = PROJECT_ROOT / "analysis/data/full_old_vs_new.csv"
BACKTEST_375K_PATH = PROJECT_ROOT / "analysis/data/proposed_l3_375k_old_vs_new.csv"
SME_SAMPLE_PATH = PROJECT_ROOT / "analysis/data/sme_sample_scored.csv"

# Join/key column (must exist in each loaded dataset)
ID_COL = "PRODUCT_ID"

# Old/new category columns used for direct comparison
OLD_LABEL_COL = "OLD_CATEGORY"
NEW_LABEL_COL = "NEW_CATEGORY"

# Optional ground-truth/SME label column (for quality metrics)
TRUTH_LABEL_COL = "SME_LABEL"

# Optional extra context columns (if present)
TEXT_COL = "TEXT_TO_EMBED"
SOURCE_TABLE_COL = "SOURCE_TABLE"

# Where to save outputs
RUN_TAG = "l3_old_vs_new_scorecard"
OUT_DIR = PROJECT_ROOT / "artifacts" / "analysis" / "comparisons" / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Known issue categories/slices to monitor closely
# Example: [("Antibodies", "Proteins & Recombinant Biomolecules"), ...]
WATCH_TRANSITIONS = []

In [ ]:
def require_columns(df: pd.DataFrame, required: list[str], df_name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} missing required columns: {missing}")


def safe_read(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{name} file not found: {path}")
    if path.suffix.lower() in {".csv", ".txt"}:
        return pd.read_csv(path)
    if path.suffix.lower() in {".parquet"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type for {name}: {path.suffix}")


def add_match_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["old_equals_new"] = out[OLD_LABEL_COL].astype(str) == out[NEW_LABEL_COL].astype(str)
    return out


def distribution_shift(df: pd.DataFrame) -> pd.DataFrame:
    old_dist = (
        df[OLD_LABEL_COL].astype(str).value_counts(normalize=True).rename("old_share")
    )
    new_dist = (
        df[NEW_LABEL_COL].astype(str).value_counts(normalize=True).rename("new_share")
    )
    joined = pd.concat([old_dist, new_dist], axis=1).fillna(0.0)
    joined["pp_shift"] = (joined["new_share"] - joined["old_share"]) * 100
    joined["abs_pp_shift"] = joined["pp_shift"].abs()
    return joined.sort_values("abs_pp_shift", ascending=False)


def transition_table(df: pd.DataFrame) -> pd.DataFrame:
    mat = (
        df.groupby([OLD_LABEL_COL, NEW_LABEL_COL], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    mat["pct_of_rows"] = mat["count"] / len(df)
    return mat


def category_churn(df: pd.DataFrame) -> pd.DataFrame:
    by_old = (
        df.groupby(OLD_LABEL_COL)
        .agg(
            rows=(ID_COL, "count"),
            changed=("old_equals_new", lambda s: int((~s).sum())),
        )
        .reset_index()
    )
    by_old["churn_rate"] = by_old["changed"] / by_old["rows"]
    return by_old.sort_values("churn_rate", ascending=False)


def quality_metrics(df: pd.DataFrame, truth_col: str, pred_col: str) -> dict:
    y_true = df[truth_col].astype(str)
    y_pred = df[pred_col].astype(str)
    return {
        "rows": int(len(df)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "report": classification_report(y_true, y_pred, output_dict=True, zero_division=0),
    }


def watch_transition_counts(df: pd.DataFrame, watch_pairs: list[tuple[str, str]]) -> pd.DataFrame:
    rows = []
    for old_cat, new_cat in watch_pairs:
        mask = (df[OLD_LABEL_COL].astype(str) == str(old_cat)) & (
            df[NEW_LABEL_COL].astype(str) == str(new_cat)
        )
        rows.append(
            {
                "old_category": str(old_cat),
                "new_category": str(new_cat),
                "count": int(mask.sum()),
                "pct_of_rows": float(mask.mean()),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
# Load datasets
full_df = safe_read(FULL_DATA_PATH, "full")
backtest_df = safe_read(BACKTEST_375K_PATH, "backtest_375k")
sme_df = safe_read(SME_SAMPLE_PATH, "sme_sample")

for name, dataf in [
    ("full", full_df),
    ("backtest_375k", backtest_df),
    ("sme_sample", sme_df),
]:
    require_columns(dataf, [ID_COL, OLD_LABEL_COL, NEW_LABEL_COL], name)

if TRUTH_LABEL_COL not in sme_df.columns:
    print(f"Warning: {TRUTH_LABEL_COL} not found in SME sample. Truth-based metrics will be skipped.")

full_df = add_match_columns(full_df)
backtest_df = add_match_columns(backtest_df)
sme_df = add_match_columns(sme_df)

print("Loaded rows:")
print({
    "full": len(full_df),
    "backtest_375k": len(backtest_df),
    "sme_sample": len(sme_df),
})

In [ ]:
# 1) FULL DATASET: stability and distribution shift
full_summary = {
    "rows": int(len(full_df)),
    "agreement_rate_old_vs_new": float(full_df["old_equals_new"].mean()),
    "churn_rate_old_vs_new": float((~full_df["old_equals_new"]).mean()),
}

full_shift = distribution_shift(full_df)
full_transitions = transition_table(full_df)
full_churn_by_category = category_churn(full_df)

print(full_summary)
print("\nTop distribution shifts (pp):")
full_shift.head(15)

In [ ]:
# 2) BACKTEST 375K: apples-to-apples migration view
backtest_summary = {
    "rows": int(len(backtest_df)),
    "agreement_rate_old_vs_new": float(backtest_df["old_equals_new"].mean()),
    "churn_rate_old_vs_new": float((~backtest_df["old_equals_new"]).mean()),
}

backtest_transitions = transition_table(backtest_df)
backtest_churn_by_category = category_churn(backtest_df)

print(backtest_summary)
print("\nTop category migrations:")
backtest_transitions.head(20)

In [ ]:
# 3) SME SAMPLE: quality-focused metrics (old vs truth, new vs truth)
sme_summary = {
    "rows": int(len(sme_df)),
    "agreement_rate_old_vs_new": float(sme_df["old_equals_new"].mean()),
}

if TRUTH_LABEL_COL in sme_df.columns:
    require_columns(sme_df, [TRUTH_LABEL_COL], "sme_sample")
    old_vs_truth = quality_metrics(sme_df, TRUTH_LABEL_COL, OLD_LABEL_COL)
    new_vs_truth = quality_metrics(sme_df, TRUTH_LABEL_COL, NEW_LABEL_COL)

    sme_summary.update(
        {
            "macro_f1_old": old_vs_truth["macro_f1"],
            "macro_f1_new": new_vs_truth["macro_f1"],
            "macro_f1_delta_new_minus_old": new_vs_truth["macro_f1"] - old_vs_truth["macro_f1"],
            "weighted_f1_old": old_vs_truth["weighted_f1"],
            "weighted_f1_new": new_vs_truth["weighted_f1"],
            "weighted_f1_delta_new_minus_old": new_vs_truth["weighted_f1"] - old_vs_truth["weighted_f1"],
        }
    )

    # Confusion matrices for quick diagnostics
    labels = sorted(pd.unique(sme_df[[TRUTH_LABEL_COL, OLD_LABEL_COL, NEW_LABEL_COL]].astype(str).values.ravel()))
    cm_old = confusion_matrix(sme_df[TRUTH_LABEL_COL].astype(str), sme_df[OLD_LABEL_COL].astype(str), labels=labels)
    cm_new = confusion_matrix(sme_df[TRUTH_LABEL_COL].astype(str), sme_df[NEW_LABEL_COL].astype(str), labels=labels)

print(sme_summary)

if TRUTH_LABEL_COL in sme_df.columns:
    print("\nTop improvements/degradations (from classification reports) are in saved JSON artifacts.")

In [ ]:
# 4) Known misclassification patterns / watch transitions
if WATCH_TRANSITIONS:
    full_watch = watch_transition_counts(full_df, WATCH_TRANSITIONS)
    backtest_watch = watch_transition_counts(backtest_df, WATCH_TRANSITIONS)
    print("Full watch transitions:")
    display(full_watch)
    print("\nBacktest watch transitions:")
    display(backtest_watch)
else:
    full_watch = pd.DataFrame()
    backtest_watch = pd.DataFrame()
    print("Set WATCH_TRANSITIONS in config to monitor known error patterns.")

In [ ]:
# Save scorecard artifacts
full_shift.to_csv(OUT_DIR / "full_distribution_shift.csv", index=True)
full_transitions.to_csv(OUT_DIR / "full_transitions.csv", index=False)
full_churn_by_category.to_csv(OUT_DIR / "full_churn_by_old_category.csv", index=False)

backtest_transitions.to_csv(OUT_DIR / "backtest_375k_transitions.csv", index=False)
backtest_churn_by_category.to_csv(OUT_DIR / "backtest_375k_churn_by_old_category.csv", index=False)

if not full_watch.empty:
    full_watch.to_csv(OUT_DIR / "full_watch_transitions.csv", index=False)
if not backtest_watch.empty:
    backtest_watch.to_csv(OUT_DIR / "backtest_375k_watch_transitions.csv", index=False)

summary_bundle = {
    "full": full_summary,
    "backtest_375k": backtest_summary,
    "sme_sample": sme_summary,
    "config": {
        "id_col": ID_COL,
        "old_label_col": OLD_LABEL_COL,
        "new_label_col": NEW_LABEL_COL,
        "truth_label_col": TRUTH_LABEL_COL,
        "watch_transitions": WATCH_TRANSITIONS,
    },
}

if TRUTH_LABEL_COL in sme_df.columns:
    summary_bundle["sme_old_vs_truth_report"] = old_vs_truth["report"]
    summary_bundle["sme_new_vs_truth_report"] = new_vs_truth["report"]

with (OUT_DIR / "scorecard_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary_bundle, f, indent=2)

print(f"Saved scorecard outputs to: {OUT_DIR}")
print("Files:")
for p in sorted(OUT_DIR.glob("*")):
    print("-", p.name)